# IND320, Assignment 3

## By: Ole Andreas Grøtting
Github: [IND320 - OLEGRO99](https://github.com/olegro99/IND320-OLEGRO99.git)\
Streamlit: https://ind320-olegro99-ybwpxf3ssrwyekrthttcuc.streamlit.app/

## Imports

In [1]:
import geopandas as gpd
import pandas as pd
import openmeteo_requests
import requests_cache
from retry_requests import retry

## retrive data

In [36]:
#Read the shapefile from atlas.nve.no
#include all files
gdf = gpd.read_file("../NVE_Region.shp")

#generate K random points within each region
K = 100 # Number of random points to generate per region
points_per_region = gdf['geometry'].sample_points(size=K, random_state=42)

# Transform WGS84 for latitude and longitude
points_wgs84 = points_per_region.to_crs(epsg=4326)

#extract the coordinates and print them
for idx, multipoint in points_wgs84.items():
    region_name = gdf.loc[idx, 'REGION']
    print(f"Region: {region_name}")
    lat = ""
    lon = ""
    for point in multipoint.geoms:
        # Do something more useful with the coordinates
        lat = lat + ","+ str(point.y)
        lon = lon + ","+ str(point.x)

gdf["AREA"] = gdf["REGION"].replace({
    "RS": "East",
    "RØ": "East",
    "RM": "North",
    "RN": "North",
    "RV": "West",
})

# Store coordinates in a dictionary
coordinates = {}

for idx, multipoint in points_wgs84.items():

    region_name = gdf.loc[idx, "AREA"]

    if region_name not in coordinates:
        coordinates[region_name] = []

    for point in multipoint.geoms:
        coordinates[region_name].append({
            "latitude": point.y,
            "longitude": point.x
        })


Region: RV
Region: RM
Region: RN
Region: RÃ
Region: RS


In [3]:
lat[1:]

'60.06884689946156,58.933265230287766,59.71190224935892,60.89806361118858,57.96669453841867,58.08496272016244,58.80918760347648,59.502145017466724,59.79182226297394,59.62541779329059,59.93733128753459,59.34323550000163,59.38533636766173,60.58911675983016,59.110319942156146,57.91193016763362,58.86916543442986,58.95469208838339,59.18905136007901,59.29493164542575,59.18325821480104,59.469265545948126,60.652907821064375,59.26717948703605,58.54624298327984,58.62644857601454,59.81863786237336,58.882551820327556,58.678211431030654,59.34993243327872,59.18559353061826,59.339783347556654,58.431351236166314,59.977618564052534,59.218476344211595,59.198326505606964,58.624933212770614,60.397703534414234,59.574068457623284,58.56476068323273,59.470460941405,58.992546897795584,59.351391610445084,58.874580434227894,59.24978429402297,58.92251693694754,57.90103278721875,58.889076254306886,58.954789352750616,60.68799940049205,58.493461528835354,59.0491160494632,60.62408830598188,59.258514472750896,57.88340

## Function for regions and timeperiods

In [37]:
def download_weather(region, start_date, end_date, batch_size=10):

    # Get coordinates for selected region
    region_coordinates = coordinates[region]

    all_data = []

    # Download coordinates in smaller batches
    for batch_start in range(0, len(region_coordinates), batch_size):

        batch = region_coordinates[batch_start:batch_start + batch_size]

        latitudes = [p["latitude"] for p in batch]
        longitudes = [p["longitude"] for p in batch]

        # API parameters
        params = {
            "latitude": latitudes,
            "longitude": longitudes,
            "start_date": start_date,
            "end_date": end_date,
            "hourly": [
                "temperature_2m",
                "precipitation",
                "relative_humidity_2m",
                "wind_speed_10m"
            ],
            "models": "era5"
        }

        # Download data
        responses = openmeteo.weather_api(
            "https://archive-api.open-meteo.com/v1/archive",
            params=params
        )

        # Process each location
        for i, response in enumerate(responses):

            hourly = response.Hourly()

            temperature = hourly.Variables(0).ValuesAsNumpy()
            precipitation = hourly.Variables(1).ValuesAsNumpy()
            humidity = hourly.Variables(2).ValuesAsNumpy()
            wind_speed = hourly.Variables(3).ValuesAsNumpy()

            time = pd.date_range(
                start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
                end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
                freq=pd.Timedelta(seconds=hourly.Interval()),
                inclusive="left"
            )

            hourly_data = pd.DataFrame({
                "time": time,
                "region": region,
                "location_id": batch_start + i,
                "latitude": response.Latitude(),
                "longitude": response.Longitude(),
                "temperature_2m": temperature,
                "precipitation": precipitation,
                "relative_humidity_2m": humidity,
                "wind_speed_10m": wind_speed
            })

            all_data.append(hourly_data)

        print(f"Downloaded {min(batch_start + batch_size, len(region_coordinates))} "
              f"/ {len(region_coordinates)} locations for {region}")

    return pd.concat(all_data, ignore_index=True)

## testing the function for all regions

In [38]:
#testing the function for East region
east_weather = download_weather(
    region="East",
    start_date="2026-10-12",
    end_date="2026-10-26"
)

print(east_weather.head())

Downloaded 10 / 100 locations for East
Downloaded 20 / 100 locations for East
Downloaded 30 / 100 locations for East
Downloaded 40 / 100 locations for East
Downloaded 50 / 100 locations for East
Downloaded 60 / 100 locations for East
Downloaded 70 / 100 locations for East
Downloaded 80 / 100 locations for East
Downloaded 90 / 100 locations for East
Downloaded 100 / 100 locations for East
                       time region  location_id  latitude  longitude  \
0 2026-10-12 00:00:00+00:00   East            0      59.0        5.0   
1 2026-10-12 01:00:00+00:00   East            0      59.0        5.0   
2 2026-10-12 02:00:00+00:00   East            0      59.0        5.0   
3 2026-10-12 03:00:00+00:00   East            0      59.0        5.0   
4 2026-10-12 04:00:00+00:00   East            0      59.0        5.0   

   temperature_2m  precipitation  relative_humidity_2m  wind_speed_10m  
0           11.00            0.0             71.281265        9.430694  
1           11.10            0

In [39]:
#testing the function for North region
north_weather = download_weather(
    region="North",
    start_date="2026-10-12",
    end_date="2026-10-26"
)

print(north_weather.head())

Downloaded 10 / 200 locations for North
Downloaded 20 / 200 locations for North
Downloaded 30 / 200 locations for North
Downloaded 40 / 200 locations for North
Downloaded 50 / 200 locations for North
Downloaded 60 / 200 locations for North
Downloaded 70 / 200 locations for North
Downloaded 80 / 200 locations for North
Downloaded 90 / 200 locations for North
Downloaded 100 / 200 locations for North
Downloaded 110 / 200 locations for North
Downloaded 120 / 200 locations for North
Downloaded 130 / 200 locations for North
Downloaded 140 / 200 locations for North
Downloaded 150 / 200 locations for North
Downloaded 160 / 200 locations for North
Downloaded 170 / 200 locations for North
Downloaded 180 / 200 locations for North
Downloaded 190 / 200 locations for North
Downloaded 200 / 200 locations for North
                       time region  location_id  latitude  longitude  \
0 2026-10-12 00:00:00+00:00  North            0     64.75      12.25   
1 2026-10-12 01:00:00+00:00  North           

In [40]:
west_weather = download_weather(
    region="West",
    start_date="2026-10-12",
    end_date="2026-10-26"
)

print(west_weather.head())

Downloaded 10 / 100 locations for West
Downloaded 20 / 100 locations for West
Downloaded 30 / 100 locations for West
Downloaded 40 / 100 locations for West
Downloaded 50 / 100 locations for West
Downloaded 60 / 100 locations for West
Downloaded 70 / 100 locations for West
Downloaded 80 / 100 locations for West
Downloaded 90 / 100 locations for West
Downloaded 100 / 100 locations for West
                       time region  location_id  latitude  longitude  \
0 2026-10-12 00:00:00+00:00   West            0      61.5       5.25   
1 2026-10-12 01:00:00+00:00   West            0      61.5       5.25   
2 2026-10-12 02:00:00+00:00   West            0      61.5       5.25   
3 2026-10-12 03:00:00+00:00   West            0      61.5       5.25   
4 2026-10-12 04:00:00+00:00   West            0      61.5       5.25   

   temperature_2m  precipitation  relative_humidity_2m  wind_speed_10m  
0          7.5125            0.2             88.669716       15.745627  
1          8.1625            0

In [ ]:
# Combine the weather data from all regions into a single DataFrame
weather_data = pd.concat(
    [east_weather, north_weather, west_weather],
    ignore_index=True
)

                       time region  location_id  latitude  longitude  \
0 2026-10-12 00:00:00+00:00   East            0      59.0        5.0   
1 2026-10-12 01:00:00+00:00   East            0      59.0        5.0   
2 2026-10-12 02:00:00+00:00   East            0      59.0        5.0   
3 2026-10-12 03:00:00+00:00   East            0      59.0        5.0   
4 2026-10-12 04:00:00+00:00   East            0      59.0        5.0   

   temperature_2m  precipitation  relative_humidity_2m  wind_speed_10m  
0           11.00            0.0             71.281265        9.430694  
1           11.10            0.1             71.300446       13.543131  
2           11.15            0.1             71.064445       14.830076  
3           11.05            0.2             72.281250       15.856356  
4           10.75            0.2             75.016472       18.610641  


In [ ]:
#printing information about the combined weather data
print(weather_data.shape)

print(weather_data["region"].value_counts())

print(weather_data.isna().sum())

print(weather_data.head())

(144000, 9)
region
North    72000
East     36000
West     36000
Name: count, dtype: int64
time                    0
region                  0
location_id             0
latitude                0
longitude               0
temperature_2m          0
precipitation           0
relative_humidity_2m    0
wind_speed_10m          0
dtype: int64
                       time region  location_id  latitude  longitude  \
0 2026-10-12 00:00:00+00:00   East            0      59.0        5.0   
1 2026-10-12 01:00:00+00:00   East            0      59.0        5.0   
2 2026-10-12 02:00:00+00:00   East            0      59.0        5.0   
3 2026-10-12 03:00:00+00:00   East            0      59.0        5.0   
4 2026-10-12 04:00:00+00:00   East            0      59.0        5.0   

   temperature_2m  precipitation  relative_humidity_2m  wind_speed_10m  
0           11.00            0.0             71.281265        9.430694  
1           11.10            0.1             71.300446       13.543131  
2          

In [45]:
for region, coords in coordinates.items():
    print(region, len(coords))

West 100
North 200
RÃ 100
East 100


In [43]:
# Download data for NO3 for 2024
data_NO3 = download_weather(
    region="NO3",
    start_date="2024-01-01",
    end_date="2024-12-31"
)

KeyError: 'NO3'